# Part A: building the news corpus

Steps:
1. Load the GDELT URL list (brewcoua/genai-gdelt-framing, CC BY 4.0)
2. Filter by month and region, then take a sample
3. Pilot: scrape 50 URLs and check how many give usable text (decision rule: 40% or more)
4. Full scrape
5. Clean (English, on-topic, no duplicates) and save `news_clean.csv`


In [1]:
# Run once
!pip install -q pandas pyarrow requests trafilatura py3langid


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\yukik\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import json, re, time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
import trafilatura
import py3langid as langid

Path("data").mkdir(exist_ok=True)

## Settings
Change these if needed. Write the values in the report.

In [ ]:
START_MONTH = "2026-04"      # first month to keep
END_MONTH   = "2026-06"      # last month to keep
REGIONS     = ["US", "UK", "EU"]
N_FULL      = 900            # URLs for the full scrape
N_PILOT     = 50             # URLs for the pilot
MAX_PER_OUTLET = 15          # so a few big sites don't dominate
MIN_WORDS   = 150            # shorter texts count as "not usable"
SEED        = 42

USER_AGENT = "NCI-student-research/1.0 (contact: x23176377@student.ncirl.ie)"

## 1. Load the URL list
About 90 MB. It downloads once and is then read from disk.

In [4]:
PARQUET = Path("data/genai_gdelt_articles.parquet")
URL = "https://huggingface.co/datasets/brewcoua/genai-gdelt-framing/resolve/main/articles.parquet"

if not PARQUET.exists():
    r = requests.get(URL, timeout=300)
    r.raise_for_status()
    PARQUET.write_bytes(r.content)

df = pd.read_parquet(PARQUET, columns=["document_id", "month", "region"])
df = df.rename(columns={"document_id": "url"})
print(len(df), "articles in the dataset")
df.head()

1116091 articles in the dataset


,url,month,region
0,https://wacotrib.com/ap/national/5-things-to-k...,2024-09,US
1,https://wobm.com/ixp/385/p/mtv-jersey-shore-vi...,2024-11,US
2,http://agenda.ge/en/news/2025/42421,2025-01,Other
3,https://www.hurriyetdailynews.com/europes-far-...,2026-04,EU
4,https://timesofindia.indiatimes.com/india/turn...,2026-04,Other


## 2. Filter and sample
Each step is counted in `steps` so the numbers can go straight into the report.

For the EU, most outlets are not in English, so only domains like `.ie`, `.eu` and `.com` are kept. Language is checked again after scraping.

In [ ]:
steps = [("all articles", len(df))]

df["month"] = df["month"].astype(str)
df = df[(df["month"] >= START_MONTH) & (df["month"] <= END_MONTH)]
steps.append((f"months {START_MONTH} to {END_MONTH}", len(df)))

df = df[df["region"].isin(REGIONS)]
steps.append(("regions " + ", ".join(REGIONS), len(df)))

# Remove tracking bits (?utm=...) so the same article isn't counted twice
df["url"] = df["url"].str.split("?").str[0].str.rstrip("/")
df = df[df["url"].str.startswith("http")].drop_duplicates("url")
steps.append(("unique URLs", len(df)))

df["outlet"] = df["url"].map(lambda u: urlsplit(u).netloc.lower().removeprefix("www."))
tld = df["outlet"].str.split(".").str[-1]
english_tlds = ["ie", "eu", "mt", "com", "net", "org", "news"]
df = df[(df["region"] != "EU") | tld.isin(english_tlds)]
steps.append(("EU: likely-English domains", len(df)))

df = df.sample(frac=1, random_state=SEED)               
df = df[df.groupby("outlet").cumcount() < MAX_PER_OUTLET]
steps.append((f"max {MAX_PER_OUTLET} per outlet", len(df)))

# Same number of URLs per region and quarter
df["quarter"] = pd.PeriodIndex(df["month"], freq="M").asfreq("Q").astype(str)
n_groups = df.groupby(["region", "quarter"]).ngroups
sample = df.groupby(["region", "quarter"]).head(N_FULL // n_groups + 1).head(N_FULL)
steps.append(("full sample", len(sample)))

pilot = sample.groupby("region").head(N_PILOT // len(REGIONS) + 1).head(N_PILOT)

sample.to_csv("data/urls_full.csv", index=False)
pilot.to_csv("data/urls_pilot.csv", index=False)

display(pd.DataFrame(steps, columns=["step", "rows"]))
pd.crosstab(sample["quarter"], sample["region"], margins=True)

,step,rows
0,all articles,1116091
1,months 2026-04 to 2026-06,76830
2,"regions US, UK, EU",37497
3,unique URLs,36959
4,EU: likely-English domains,27039
5,max 15 per outlet,21996
6,full sample,900


region,EU,UK,US,All
quarter,,,,
2026Q2,301,298,301,900
All,301,298,301,900


## 3. Scraping function


Results are saved to `data/scraped.csv` every 25 URLs. URLs already in that file are skipped.

In [6]:
HEADERS = {"User-Agent": USER_AGENT}
SCRAPED = Path("data/scraped.csv")
PAYWALL = re.compile(r"subscribe (now|to continue|to read)|already a subscriber|sign in to continue", re.I)
robots_cache = {}

def robots_ok(url):
    site = "https://" + urlsplit(url).netloc
    if site not in robots_cache:
        rp = RobotFileParser()
        try:
            r = requests.get(site + "/robots.txt", headers=HEADERS, timeout=10)
            rp.parse(r.text.splitlines() if r.status_code == 200 else [])
        except requests.RequestException:
            rp.parse([])
        robots_cache[site] = rp
    return robots_cache[site].can_fetch(USER_AGENT, url)

def scrape(url):
    result = {"url": url, "status": "", "title": "", "date": "", "text": "",
              "n_words": 0, "lang": "", "accessed": datetime.now(timezone.utc).isoformat()}
    if not robots_ok(url):
        result["status"] = "robots_blocked"
        return result
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
    except requests.RequestException:
        result["status"] = "failed"
        return result
    if r.status_code != 200:
        result["status"] = f"http_{r.status_code}"
        return result

    out = trafilatura.extract(r.text, output_format="json", with_metadata=True,
                              include_comments=False, include_tables=False)
    if not out:
        result["status"] = "no_text"
        return result
    out = json.loads(out)
    text = out.get("text") or ""
    result.update(title=out.get("title") or "", date=out.get("date") or "",
                  text=text, n_words=len(text.split()), lang=langid.classify(text)[0])

    if result["n_words"] < MIN_WORDS:
        result["status"] = "too_short"
    elif result["n_words"] < 400 and PAYWALL.search(text):
        result["status"] = "paywall"
    elif result["lang"] != "en":
        result["status"] = "not_english"
    else:
        result["status"] = "ok"
    return result

def scrape_all(urls):
    done = pd.read_csv(SCRAPED, keep_default_na=False) if SCRAPED.exists() else pd.DataFrame(columns=["url"])
    todo = [u for u in urls if u not in set(done["url"])]
    print(len(urls) - len(todo), "already done,", len(todo), "to go")
    new = []
    for i, url in enumerate(todo, 1):
        new.append(scrape(url))
        time.sleep(1)                                   # be polite
        if i % 25 == 0 or i == len(todo):
            done = pd.concat([done, pd.DataFrame(new)], ignore_index=True)
            done.to_csv(SCRAPED, index=False)
            new = []
            print(f"{i}/{len(todo)}")
    return done[done["url"].isin(urls)]

## 4. Pilot: 50 URLs, then the decision (Day 2)

In [7]:
res = scrape_all(list(pilot["url"]))
res = res.merge(pilot[["url", "region"]], on="url")

print(res["status"].value_counts(), "\n")
print(res.groupby("region")["status"].apply(lambda s: (s == "ok").mean()).round(2))

rate = (res["status"] == "ok").mean()
print(f"\nUsable: {rate:.0%}")
print("Decision:", "scale up to the full list" if rate >= 0.40 else "below 40%, use the GDELT fallback (section 6)")

1 already done, 49 to go
25/49
49/49
status
ok                27
not_english       14
http_403           3
robots_blocked     3
http_404           2
http_429           1
Name: count, dtype: int64 

region
EU    0.47
UK    0.38
US    0.76
Name: status, dtype: float64

Usable: 54%
Decision: scale up to the full list


## 5. Full scrape (Day 3)
This takes roughly 20–40 minutes. The pilot URLs are not fetched again.

In [8]:
res = scrape_all(list(sample["url"]))
print(res["status"].value_counts())

60 already done, 840 to go
25/840
50/840
75/840
100/840
125/840
150/840
175/840
200/840
225/840
250/840
275/840
300/840
325/840
350/840
375/840
400/840
425/840
450/840
475/840
500/840
525/840
550/840
575/840
600/840
625/840
650/840
675/840
700/840
725/840
750/840
775/840
800/840
825/840
840/840
status
ok                540
not_english       158
http_403           58
too_short          34
http_404           32
robots_blocked     23
http_429           22
http_451           14
no_text             6
failed              4
http_406            3
http_410            3
http_202            2
http_503            1
Name: count, dtype: int64


## 6. (Only if the pilot was below 40%) GDELT fallback
The GDELT DOC API returns up to 250 articles per request and only covers the last ~3 months. Note this in the report, because these articles come from a different time window. Running section 5 again then scrapes the new URLs.

In [9]:
RUN_FALLBACK = False

if RUN_FALLBACK:
    query = '("AI regulation" OR "AI Act" OR "AI law" OR "regulate AI" OR "AI safety") sourcelang:english'
    extra = []
    for country, region in [("US", "US"), ("UK", "UK"), ("EI", "EU")]:   # EI = Ireland
        r = requests.get("https://api.gdeltproject.org/api/v2/doc/doc",
                         params={"query": f"{query} sourcecountry:{country}", "mode": "ArtList",
                                 "format": "json", "maxrecords": 250, "timespan": "3months"},
                         timeout=60)
        for a in r.json().get("articles", []):
            extra.append({"url": a["url"].split("?")[0].rstrip("/"), "region": region,
                          "month": a["seendate"][:4] + "-" + a["seendate"][4:6],
                          "outlet": a["domain"].removeprefix("www.")})
        time.sleep(6)                                    # GDELT asks for max 1 request / 5 s
    extra = pd.DataFrame(extra).drop_duplicates("url")
    extra["quarter"] = pd.PeriodIndex(extra["month"], freq="M").asfreq("Q").astype(str)
    sample = pd.concat([sample, extra]).drop_duplicates("url")
    print(len(extra), "extra URLs; run section 5 again")

## 7. Clean and save (Day 3)
- keep only usable texts (`ok`)
- drop very long pages (> 3,000 words, usually live blogs)
- **on-topic:** at least 2 AI words and at least 2 regulation words. This list is kept separate from the frame keywords, so the filter does not decide which frames appear.
- drop duplicates (same text, or same title from the same wire story)

In [ ]:
AI_WORDS  = re.compile(r"\b(AI|artificial intelligence|chatbots?|ChatGPT|large language models?|LLMs?|OpenAI|Gemini)\b")
REG_WORDS = re.compile(r"\b(regulat\w*|legislat\w*|laws?|AI Act|bills?|rules?|oversight|ban|governance|policy|lawmakers?)\b", re.I)

news = res.merge(sample[["url", "outlet", "region", "month", "quarter"]], on="url")
clean_steps = [("scraped", len(news))]

news = news[news["status"] == "ok"]
clean_steps.append(("usable text", len(news)))

news = news[news["n_words"] <= 3000]
clean_steps.append(("<= 3000 words", len(news)))

ai_hits  = news["text"].map(lambda t: len(AI_WORDS.findall(t)))
reg_hits = news["text"].map(lambda t: len(REG_WORDS.findall(t)))
news = news[(ai_hits >= 2) & (reg_hits >= 2)]
clean_steps.append(("on-topic", len(news)))

news["text_key"] = news["text"].str.lower().str.replace(r"\W+", " ", regex=True).str.strip()
news = news.drop_duplicates("text_key")
news = news[(news["title"] == "") | ~news.duplicated("title")]   
clean_steps.append(("no duplicates", len(news)))

news["date"] = news["date"].where(news["date"] != "", news["month"])
news["n_words"] = news["n_words"].astype(int)
news = news.reset_index(drop=True)
news["doc_id"] = [f"news_{i:04d}" for i in range(1, len(news) + 1)]
news = news[["doc_id", "text", "outlet", "title", "date", "month", "quarter", "region", "url", "n_words", "accessed"]]
news.to_csv("data/news_clean.csv", index=False)

display(pd.DataFrame(clean_steps, columns=["step", "rows"]))
print(news["outlet"].nunique(), "outlets, median", int(news["n_words"].median()), "words")
pd.crosstab(news["quarter"], news["region"], margins=True)

,step,rows
0,scraped,900
1,usable text,540
2,<= 3000 words,514
3,on-topic,47
4,no duplicates,45


38 outlets, median 1009 words


region,EU,UK,US,All
quarter,,,,
2026Q2,10,19,16,45
All,10,19,16,45


In [11]:
check = res[res["status"] == "ok"].copy()
check["ai"] = check["text"].map(lambda t: len(AI_WORDS.findall(t)))
check["reg"] = check["text"].map(lambda t: len(REG_WORDS.findall(t)))
print("AI words >= 2:", (check["ai"] >= 2).mean().round(2), "| reg words >= 2:", (check["reg"] >= 2).mean().round(2))
check.loc[check["ai"] < 2, ["title", "n_words"]].head(15)

AI words >= 2: 0.24 | reg words >= 2: 0.31


,title,n_words
88,Timeless Classics: The Best Time Travel Movies...,2235.0
286,'Speechless' Cher discovers she has secret gra...,473.0
742,Celebrities and reality stars pay tribute to T...,450.0
901,Church Of Scientology Slams Viral 'Speed Runni...,496.0
902,"Michael Tilson Thomas, renowned conductor and ...",1668.0
904,"Florida Woman Accused Of Squatting, Selling Ho...",209.0
905,Luzerne County Council advances revised non-di...,497.0
906,House’s budget plan brushes aside Republican p...,1220.0
909,Trump names housing finance chief Pulte as act...,333.0
910,Records detail spending under Windsor County s...,1351.0


## 8. Log for the report
Saves the settings, filter counts and access dates for the data section.

In [12]:
log = {
    "dataset": "brewcoua/genai-gdelt-framing (CC BY 4.0)",
    "settings": {"months": [START_MONTH, END_MONTH], "regions": REGIONS, "n_full": N_FULL,
                 "max_per_outlet": MAX_PER_OUTLET, "min_words": MIN_WORDS, "seed": SEED},
    "selection_steps": steps,
    "cleaning_steps": clean_steps,
    "access_dates": [news["accessed"].min(), news["accessed"].max()],
    "on_topic_rule": ">= 2 AI words and >= 2 regulation words",
}
Path("data/corpus_log.json").write_text(json.dumps(log, indent=2, default=str))
log

{'dataset': 'brewcoua/genai-gdelt-framing (CC BY 4.0)',
 'settings': {'months': ['2026-04', '2026-06'],
  'regions': ['US', 'UK', 'EU'],
  'n_full': 900,
  'max_per_outlet': 15,
  'min_words': 150,
  'seed': 42},
 'selection_steps': [('all articles', 1116091),
  ('months 2026-04 to 2026-06', 76830),
  ('regions US, UK, EU', 37497),
  ('unique URLs', 36959),
  ('EU: likely-English domains', 27039),
  ('max 15 per outlet', 21996),
  ('full sample', 900)],
 'cleaning_steps': [('scraped', 900),
  ('usable text', 540),
  ('<= 3000 words', 514),
  ('on-topic', 47),
  ('no duplicates', 45)],
 'access_dates': ['2026-10-19T17:34:31.534393+00:00',
  '2026-10-20T12:53:14.577565+00:00'],
 'on_topic_rule': '>= 2 AI words and >= 2 regulation words'}

**Licensing note:** the article text belongs to the publishers. so we will keep `scraped.csv` and `news_clean.csv` out of the submission ZIP. we will share `urls_full.csv`, this notebook and `corpus_log.json` so others can rebuild the corpus.